In [ ]:
from eval import wikitq_match_func_for_samples
import re
import ast 
import pickle
import json
import os
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict

In [ ]:
def process_pred_text(text, precision_threshold=10):
    if text is None:
        return ""
    s = str(text).replace("*" ,"").replace(".." , "")
    s = s.replace("True", "yes").replace("true", "yes")
    s = s.replace("False", "no").replace("false", "no")
    #Because calculations are involved, carry-over is required.
    def shrink_decimal(match):
        full_num_str = match.group(0)
        if "." in full_num_str:
            integer_part, decimal_part = full_num_str.split(".")
            if len(decimal_part) > precision_threshold:
                try:
                    return "{:.2f}".format(float(full_num_str))
                except ValueError:
                    return full_num_str
        return full_num_str
    s = re.sub(r"[-+]?\d*\.\d+", shrink_decimal, s)
    return s

def read_jsonl_to_list(file_path):

    data_list = []
    
    if not os.path.exists(file_path):
        print(f"Error: file not exist - {file_path}")
        return data_list

    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line: continue
            
            try:
                item = json.loads(line)
                item_id = item.get("ids", None)
                true_answer = item.get("true_answer", "")
                raw_first = item.get("first_result", "")
                processed_first = process_pred_text(raw_first)

               
                raw_finals = item.get("final_result", [])
                if isinstance(raw_finals, list):
                    processed_finals = [process_pred_text(res) for res in raw_finals]
                else:
                    processed_finals = [process_pred_text(raw_finals)]
                
                
                data_list.append({
                    "id": item_id,
                    "first": processed_first,
                    "finals": processed_finals,
                    "gold": true_answer
                })
                
            except Exception as e:
                print(f"Error processing line: {e}")

    return data_list  

def evaluate_task_result(result_file_path, start=-1, over=-1):
    all_data = read_jsonl_to_list(result_file_path)
    if not all_data:
        return

    filtered_data = []
    for item in all_data:
        try:
            id_int = int(item['id'].split("-")[1])
            if (start != -1 and id_int < start) or (over != -1 and id_int > over):
                continue
            filtered_data.append(item)
        except:
            continue

    samples_first = [[item['id'], item['first'], item['gold']] for item in filtered_data]
    try:
        acc_first, error1 = wikitq_match_func_for_samples(samples_first)
        print(f"[Length] : {len(filtered_data)}")
        print(f"🚀 [Initial Result]  Accuracy: {acc_first:.10%}")
    except Exception as e:
        print(f"❌ [Initial] Error: {e}")

  
    max_refine_steps = 0
    if filtered_data:
        max_refine_steps = max(len(item['finals']) for item in filtered_data)
    

    for step_idx in range(max_refine_steps):
        samples_step = []
        for item in filtered_data:
            res_list = item['finals']
            
            if len(res_list) == 0:
                res_list = ["Not Available"]
            
            current_res = res_list[step_idx] if step_idx < len(res_list) else res_list[-1]
            samples_step.append([item['id'], current_res, item['gold']])
        
        try:
            acc_step, error2 = wikitq_match_func_for_samples(samples_step)
            print(f"🔄 [Refine Step {step_idx + 1}]  Accuracy: {acc_step:.10%}")
        except Exception as e:
            print(f"❌ [Refine {step_idx + 1}] Error: {e}")

    print("=" * 70)
    
    return samples_first , samples_step

#create result path
def create_path(num , model_name , type , route_type):
    return  f"../result/run{num}_results/wikitq/{model_name}/wikitq-{model_name}-cols_{type}-routing_{route_type}/result.jsonl"
    

# How to eval

You can choose to directly pass in the results.josn file, or fill in the relevant configuration in the configuration file if you do not change result path in config.
The parameters for create_path are as follows:
* **num**: task_run_ids in config
* **model_name**: model_name in config
* **type**: col_select_type in config
* **route_type**: routing_strategy in config

In [ ]:
# example
path = create_path(1 , "gpt-4o-mini"  , "entity"  , "hybrid")
entity_first , entity_final = evaluate_task_result(path)